# Exploratory Data Analysis of All Data 

This comprehensively covers every dataset that will be used for training/tuning.

## Table of Contents

- [Biohub Cell Tracking Dataset - Base Kaggle](#biohub-cell-tracking-dataset---base-kaggle)

## Biohub Cell Tracking Dataset - Base Kaggle

In [ ]:
from pathlib import Path
from typing import Any, cast

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import zarr
from IPython.display import HTML
from matplotlib import animation, colormaps
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Line3DCollection

plt.rcParams["animation.embed_limit"] = 100

### Data Loading and Visual Inspection

Initial visual pass of the Base Data shows that there is a fair high number of irregularities in track procession. Observations indicate clear jumps in some portions of a multitude of tracks. Based on visual inspection, I conclude that these jumps are a consequence of irregular camera framing, and not bad data. More concretely, inspecting the 5 most irregular tracks from each embryo (44b6 and 6bba) shows that one track in particular (6bba_f20478e9) contains a massive z-slice jump. In the multiview overlay where we can see both the top view (z-mip) and the side view (x-mip), the tracked cells clearly show this jump. Another anomoly observed here is that the video seems to "stall" on one frame prior to the jump, t=61, then after four frames there is a sudden jump at t-65, before correcting back to original z-position at t=66. My best guess for these tracks is that there is an error with how the data is recorded? In any case, my general remmediation steps going forward will be to do a semi-undo of the "shaky cam" effect seen in a large percentage of the videos. For the 6bba_f20478e9 video, we will simply split this video into two separate videos/tracks, one before the jump and one after. This way we retain as much data as possible.

In [ ]:
# Create data paths to train and test data
DATA_DIR = Path("../../data/competition")

train_zarrs = sorted((DATA_DIR / "train").glob("*.zarr"))
train_geffs = sorted((DATA_DIR / "train").glob("*.geff"))
test_zarrs = sorted((DATA_DIR / "test").glob("*.zarr"))

names = [p.stem for p in train_zarrs]
assert names == [p.stem for p in train_geffs], "every train volume needs a paired GEFF"
print(f"{len(names)} train videos (zarr+geff pairs), {len(test_zarrs)} test zarr volumes")
print("embryos:", sorted({n.split("_")[0] for n in names}), "| First entry:", names[0])

In [ ]:
# video split by embryo
embryo_counts = pl.DataFrame({"embryo": [n.split("_")[0] for n in names]}).group_by("embryo").len().sort("embryo")
embryos = embryo_counts["embryo"].to_list()
counts = embryo_counts["len"].to_list()

fig, ax = plt.subplots(figsize=(5, 5))
ax.bar(embryos, counts)
for i, c in enumerate(counts):
    ax.text(i, c, str(c), ha="center", va="bottom")

ax.set_xlabel("embryo")
ax.set_ylabel("train videos")
ax.set_title(f"{len(names)} train videos from {len(embryos)} embryos")
plt.show()

In [ ]:
# Volumes are zarr v3 groups holding one array "0": (T, Z, Y, X) uint16, one timepoint per chunk
SCALE_TZYX = (1.0, 1.625, 0.40625, 0.40625)  # s / µm per voxel, from the OME multiscales metadata

name = names[0]
root = zarr.open_group(DATA_DIR / "train" / f"{name}.zarr", mode="r")
volume = root["0"]
assert isinstance(volume, zarr.Array)  # zarr getitem returns Array | Group; narrow it

ome: Any = root.attrs["multiscales"]  # JSON metadata; index freely
ome_scale = tuple(ome[0]["datasets"][0]["coordinateTransformations"][0]["scale"])
assert ome_scale == SCALE_TZYX, f"unexpected voxel scale {ome_scale}"

stats: Any = root.attrs["image_statistics"]
print("dataset:", name)
print("shape:", volume.shape, "| dtype:", volume.dtype, "| chunks:", volume.chunks)
print("intensity quantiles:", stats["quantiles"])

In [ ]:
def _read_array(g: zarr.Group, key: str) -> np.ndarray:
    arr = g[key]
    assert isinstance(arr, zarr.Array)
    return np.asarray(arr[:])


def load_geff(path: Path) -> tuple[pl.DataFrame, pl.DataFrame, dict[str, Any]]:
    """Read a GEFF annotation store into (nodes, edges, metadata) with plain zarr.

    Layout: nodes/ids + nodes/props/{t,z,y,x}/values, edges/ids as (source, target)
    pairs, plus a "geff" attrs dict whose extra.estimated_number_of_nodes is the
    total cell count the metric's node-count penalty compares against.
    """
    g = zarr.open_group(path, mode="r")
    nodes = pl.DataFrame(
        {
            "node_id": _read_array(g, "nodes/ids"),
            "t": _read_array(g, "nodes/props/t/values"),
            "z": _read_array(g, "nodes/props/z/values"),
            "y": _read_array(g, "nodes/props/y/values"),
            "x": _read_array(g, "nodes/props/x/values"),
        }
    )
    edge_arr = _read_array(g, "edges/ids")
    edges = pl.DataFrame({"source_id": edge_arr[:, 0], "target_id": edge_arr[:, 1]})
    meta = cast(dict[str, Any], g.attrs["geff"])
    return nodes, edges, meta


nodes, edges, meta = load_geff(DATA_DIR / "train" / f"{name}.geff")
estimated_total = meta["extra"]["estimated_number_of_nodes"]
n_divisions = edges.group_by("source_id").len().filter(pl.col("len") == 2).height
print(f"{name}: {len(nodes)} annotated nodes (~{len(nodes) / estimated_total:.1%} of est. {estimated_total} cells)")
print(f"{len(edges)} edges, {n_divisions} divisions (parents with 2 outgoing edges)")
nodes.head()

In [ ]:
# check GEFF coordinates index in the zarr array directly as [t, z, y, x]
t_show = int(np.median(nodes["t"].to_numpy()))
frame = np.asarray(volume[t_show])  # single chunk read: (Z, Y, X)
mip = frame.max(axis=0)  # max-intensity projection over Z

sel = nodes.filter(pl.col("t") == t_show)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mip, cmap="gray")
ax.scatter(sel["x"].to_numpy(), sel["y"].to_numpy(), s=50, facecolors="none", edgecolors="lime")
ax.set_title(f"{name} — t={t_show}, Z-MIP + {len(sel)} annotated cells")
plt.show()

In [ ]:
# Z-MIP per frame + lineage tracks animated
T = volume.shape[0]
mips = np.stack([np.asarray(volume[t]).max(axis=0) for t in range(T)])
pts_by_t = [nodes.filter(pl.col("t") == t).select(["x", "y"]).to_numpy() for t in range(T)]

# xy[i] is the (x, y) pixel of node i
xy = nodes.select(["x", "y"]).to_numpy()
idx = {n: i for i, n in enumerate(nodes["node_id"].to_list())}
src = [idx[s] for s in edges["source_id"].to_list()]
tgt = [idx[t] for t in edges["target_id"].to_list()]
segments = xy[np.array([src, tgt]).T]  # (E, 2, 2) segments (x0, y0) -> (x1, y1)

# a unique cell = a branch chain, starts at a root or at a division
kids: dict[int, list[int]] = {}
for s_i, t_i in zip(src, tgt, strict=True):
    kids.setdefault(s_i, []).append(t_i)

daughters = set(tgt)
starts = [i for i in range(len(xy)) if i not in daughters]
for ks in kids.values():
    if len(ks) == 2:  # both start a new cell on division
        starts.extend(ks)

track = np.full(len(xy), -1)
for tid, start in enumerate(starts):
    i = start
    while track[i] < 0:
        track[i] = tid
        ks = kids.get(i, [])
        if len(ks) != 1:
            break
        i = ks[0]

assert (track >= 0).all(), "every node must belong to a track"

cmap = colormaps["tab20"]
track_color = [cmap(t * 2 % 20) for t in range(len(starts))]

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(mips[0], cmap="gray")
ax.add_collection(LineCollection(list(segments), colors=[track_color[track[i]] for i in src], linewidths=1.0))
ax.scatter(xy[:, 0], xy[:, 1], s=12, c=[track_color[t] for t in track])
sc = ax.scatter([], [], s=50, facecolors="none", edgecolors="lime")  # nodes at current t
for tid, start in enumerate(starts):
    ax.text(xy[start, 0] + 2, xy[start, 1] + 2, f"cell {tid}", color=track_color[tid], fontsize=8)

ax.set_axis_off()
fig.suptitle(f"{name} — {len(starts)} unique cells tracked")
title = ax.set_title("")


def update(t: int) -> None:
    im.set_data(mips[t])
    sc.set_offsets(pts_by_t[t])
    title.set_text(f"{name} — t={t}/{T - 1} — {len(pts_by_t[t])}/{len(xy)} nodes")


anim = animation.FuncAnimation(fig, update, frames=T, interval=100, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Motion of every annotated track, split by embryo.
# Each edge drawn as a 3D segment in µm, every track recentered to start at the origin.
# Segments revealed in time order, a segment far longer than its neighbors is an erratic jump
seg_um = {}  # per embryo: (E, 2, 3) segments
t_srcs = {}  # per embryo: source timepoint of each segment
lids = {}  # per embryo: globally unique lineage id of each segment
vid_names = {}  # per embryo: sorted video stems
vids = {}  # per embryo: video index of each segment
cuts = {}  # per embryo: how many segments are visible at each frame

for embryo in embryos:
    segs = []
    t_src = []
    lid_list = []
    vid_list = []
    next_lid = 0
    gpaths = sorted((DATA_DIR / "train").glob(f"{embryo}_*.geff"))
    vid_names[embryo] = [p.stem for p in gpaths]
    for v, gpath in enumerate(gpaths):
        # Pull data from the iterated items
        nds, eds, _ = load_geff(gpath)
        idx = {n: i for i, n in enumerate(nds["node_id"].to_list())}
        scale_xyz = (SCALE_TZYX[3], SCALE_TZYX[2], SCALE_TZYX[1])  # µm per voxel in x, y, z
        pos = nds.select(["x", "y", "z"]).to_numpy() * scale_xyz
        tvals = nds["t"].to_numpy()
        src = [idx[s] for s in eds["source_id"].to_list()]
        tgt = [idx[t] for t in eds["target_id"].to_list()]

        # recenter each lineage (a founder + all its descendants) on the founder's position
        kids = {}
        for s_i, t_i in zip(src, tgt, strict=True):
            kids.setdefault(s_i, []).append(t_i)

        daughters = set(tgt)
        roots = [i for i in range(len(pos)) if i not in daughters]
        lineage = np.full(len(pos), -1)
        for lid, root in enumerate(roots):
            stack = [root]
            while stack:
                i = stack.pop()
                lineage[i] = lid
                stack.extend(kids.get(i, []))

        assert (lineage >= 0).all(), "every node must belong to a lineage"
        pos -= pos[np.array(roots)[lineage]]

        # put data into the lists
        for s_i, t_i in zip(src, tgt, strict=True):
            segs.append(pos[[s_i, t_i]])
            t_src.append(tvals[s_i])
            lid_list.append(int(lineage[s_i]) + next_lid)
            vid_list.append(v)

        next_lid += len(roots)

    order = np.argsort(t_src)
    seg_um[embryo] = np.stack(segs)[order]
    t_srcs[embryo] = np.asarray(t_src)[order]
    lids[embryo] = np.asarray(lid_list)[order]
    vids[embryo] = np.asarray(vid_list)[order]
    cuts[embryo] = np.searchsorted(t_srcs[embryo], np.arange(T), side="right")
    steps = np.linalg.norm(np.diff(seg_um[embryo], axis=1)[:, 0], axis=1)
    print(
        f"{embryo}: {len(segs)} track steps — "
        f"median {np.median(steps):.2f} µm, p99 {np.percentile(steps, 99):.2f} µm, max {steps.max():.2f} µm"
    )

fig = plt.figure(figsize=(16, 10), dpi=80)
axs = []
colls = []
for i, embryo in enumerate(embryos):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    assert isinstance(ax, Axes3D)
    coll = Line3DCollection([], color=f"C{i}", linewidths=0.5)
    ax.add_collection3d(coll, autolim=False)  # limits are set manually below
    lim = float(np.abs(seg_um[embryo]).max())
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.scatter(0, 0, 0, color="black", s=20)  # shared start of every track
    ax.set_xlabel("Δx (µm)")
    ax.set_ylabel("Δy (µm)")
    ax.set_zlabel("Δz (µm)")
    ax.set_title(f"embryo {embryo}")
    axs.append(ax)
    colls.append(coll)


def update_tracks(f: int) -> None:
    for coll, embryo in zip(colls, embryos, strict=True):
        coll.set_segments(list(seg_um[embryo][: cuts[embryo][f]]))

    for ax in axs:
        ax.view_init(elev=25, azim=-60 + f * 1.8)  # slow rotation reveals z-jumps

    fig.suptitle(f"all track motion by embryo — t={f}/{T - 1}")


anim_tracks = animation.FuncAnimation(fig, update_tracks, frames=T, interval=150, blit=False)
plt.close(fig)
HTML(anim_tracks.to_jshtml())

In [ ]:
data = []
for gpath in sorted((DATA_DIR / "train").glob(f"{embryos[0]}_*.geff")):
    nds, eds, _ = load_geff(gpath)
    data.append(_)

df_44b6 = pl.DataFrame(data)
print(df_44b6)

data = []
for gpath in sorted((DATA_DIR / "train").glob(f"{embryos[1]}_*.geff")):
    nds, eds, _ = load_geff(gpath)
    data.append(_)

df_6bba = pl.DataFrame(data)
print(df_6bba)

In [ ]:
# per embryo, the TOP_N most erratic lineages (ranked by their longest step).
# steps at/above p95 within them drawn red.
TOP_N = 100
norm_um = {}
jump_um = {}
norm_cuts = {}
jump_cuts = {}
for embryo in embryos:
    steps = np.linalg.norm(np.diff(seg_um[embryo], axis=1)[:, 0], axis=1)
    thr = float(np.percentile(steps, 95))
    lin_max = (
        pl.DataFrame({"lid": lids[embryo], "step": steps})
        .group_by("lid")
        .agg(pl.col("step").max())
        .sort("step", descending=True)
    )
    keep = set(lin_max.head(TOP_N)["lid"].to_list())
    in_kept = np.isin(lids[embryo], list(keep))
    is_jump = in_kept & (steps >= thr)
    norm_um[embryo] = seg_um[embryo][in_kept & ~is_jump]
    norm_cuts[embryo] = np.searchsorted(t_srcs[embryo][in_kept & ~is_jump], np.arange(T), side="right")
    jump_um[embryo] = seg_um[embryo][is_jump]
    jump_cuts[embryo] = np.searchsorted(t_srcs[embryo][is_jump], np.arange(T), side="right")
    print(
        f"{embryo}: top {TOP_N} lineages by max step (max >= {lin_max['step'][TOP_N - 1]:.2f} µm) — "
        f"{int(in_kept.sum())} segments, {int(is_jump.sum())} red p95+ jumps (thr = {thr:.2f} µm)"
    )

fig = plt.figure(figsize=(16, 10), dpi=80)
axs2 = []
norm_colls = []
jump_colls = []
for i, embryo in enumerate(embryos):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    assert isinstance(ax, Axes3D)
    nc = Line3DCollection([], color=f"C{i}", linewidths=0.5)
    jc = Line3DCollection([], color="red", linewidths=1.5)
    ax.add_collection3d(nc, autolim=False)
    ax.add_collection3d(jc, autolim=False)
    lim = float(np.abs(seg_um[embryo]).max())
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.scatter(0, 0, 0, color="black", s=20)  # shared start of every track
    ax.set_xlabel("Δx (µm)")
    ax.set_ylabel("Δy (µm)")
    ax.set_zlabel("Δz (µm)")
    ax.set_title(f"embryo {embryo} — top {TOP_N} most erratic lineages")
    axs2.append(ax)
    norm_colls.append(nc)
    jump_colls.append(jc)


def update_flagged(f: int) -> None:
    for i, embryo in enumerate(embryos):
        norm_colls[i].set_segments(list(norm_um[embryo][: norm_cuts[embryo][f]]))
        jump_colls[i].set_segments(list(jump_um[embryo][: jump_cuts[embryo][f]]))

    for ax in axs2:
        ax.view_init(elev=25, azim=-60 + f * 1.8)

    fig.suptitle(f"top-{TOP_N} most erratic lineages (p95+ steps red) — t={f}/{T - 1}")


anim_flagged = animation.FuncAnimation(fig, update_flagged, frames=T, interval=150, blit=False)
plt.close(fig)
HTML(anim_flagged.to_jshtml())

Viewing the top 5 most erratic tracks as videos for each embryo

In [ ]:
# the N_SHOW most erratic lineages per embryo, each replayed as its own video
# segments revealed in time order, p95+ steps red
N_SHOW = 5


def _steps(segs: np.ndarray) -> np.ndarray:
    return np.linalg.norm(np.diff(segs, axis=1)[:, 0], axis=1)


tracks: list[dict[str, Any]] = []  # one entry per subplot
for r, embryo in enumerate(embryos):
    all_steps = _steps(seg_um[embryo])
    thr = float(np.percentile(all_steps, 95))
    lin_max = (
        pl.DataFrame({"lid": lids[embryo], "step": all_steps})
        .group_by("lid")
        .agg(pl.col("step").max())
        .sort("step", descending=True)
    )
    for rank, lid in enumerate(lin_max.head(N_SHOW)["lid"].to_list()):
        m = lids[embryo] == lid
        t_segs, t_t = seg_um[embryo][m], t_srcs[embryo][m]
        red = _steps(t_segs) >= thr
        tracks.append(
            {
                "embryo": embryo,
                "rank": rank,
                "row": r,
                "max_step": float(_steps(t_segs).max()),
                "video": vid_names[embryo][vids[embryo][m][0]],
                "t_lo": int(t_t.min()),
                "t_hi": int(t_t.max()),
                "segs": t_segs,
                "norm": t_segs[~red],
                "jump": t_segs[red],
                "cuts": np.searchsorted(t_t, np.arange(T), side="right"),
                "norm_cuts": np.searchsorted(t_t[~red], np.arange(T), side="right"),
                "jump_cuts": np.searchsorted(t_t[red], np.arange(T), side="right"),
            }
        )

fig = plt.figure(figsize=(20, 8), dpi=80)
for j, tr in enumerate(tracks):
    ax = fig.add_subplot(2, N_SHOW, j + 1, projection="3d")
    assert isinstance(ax, Axes3D)
    nc = Line3DCollection([], color=f"C{tr['row']}", linewidths=0.8)
    jc = Line3DCollection([], color="red", linewidths=2.0)
    ax.add_collection3d(nc, autolim=False)
    ax.add_collection3d(jc, autolim=False)
    (head,) = ax.plot([], [], [], "o", color="black", markersize=4)
    lim = float(np.abs(tr["segs"]).max()) * 1.05
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.scatter(0, 0, 0, color="gray", s=10)  # track start
    ax.set_title(
        f"{tr['embryo']} #{tr['rank'] + 1} — max {tr['max_step']:.1f} µm\n{tr['video']}, t={tr['t_lo']}..{tr['t_hi']}",
        fontsize=8,
    )
    tr["ax"] = ax
    tr["nc"] = nc
    tr["jc"] = jc
    tr["head"] = head


def update_erratic(f: int) -> None:
    for tr in tracks:
        tr["nc"].set_segments(list(tr["norm"][: tr["norm_cuts"][f]]))
        tr["jc"].set_segments(list(tr["jump"][: tr["jump_cuts"][f]]))
        k = tr["cuts"][f]
        if k > 0:
            x, y, z = tr["segs"][k - 1, 1]
            tr["head"].set_data([x], [y])
            tr["head"].set_3d_properties([z])
        else:
            tr["head"].set_data([], [])
            tr["head"].set_3d_properties([])
        tr["ax"].view_init(elev=25, azim=-60 + f * 1.8)

    fig.suptitle(f"top {N_SHOW} most erratic lineages per embryo — t={f}/{T - 1}")


anim_erratic = animation.FuncAnimation(fig, update_erratic, frames=T, interval=150, blit=False)
plt.close(fig)
HTML(anim_erratic.to_jshtml())

In [ ]:
# check of the 6bba_f20478e9 identity errors.
# (t, z) trace of each affected lineage to inspect what's actually happening with the jump
# the red node is a one-frame mismatch
bn, be, _ = load_geff(DATA_DIR / "train" / "6bba_f20478e9.geff")
bidx = {n: i for i, n in enumerate(bn["node_id"].to_list())}
bsrc = [bidx[s] for s in be["source_id"].to_list()]
btgt = [bidx[t] for t in be["target_id"].to_list()]
bkids: dict[int, list[int]] = {}
bparent: dict[int, int] = {}
for s_i, t_i in zip(bsrc, btgt, strict=True):
    bkids.setdefault(s_i, []).append(t_i)
    bparent[t_i] = s_i

bt = bn["t"].to_numpy()
bz = bn["z"].to_numpy()

# z-residual vs interpolated t-1/t+1 neighbors, the 4 worst are the mismatched nodes
bres = np.zeros(len(bn))
for i in range(len(bn)):
    p = bparent.get(i)
    if p is None or len(bkids.get(i, [])) != 1:
        continue
    n_i = bkids[i][0]
    if bt[n_i] - bt[p] != 2:
        continue
    bres[i] = abs(bz[i] - (bz[p] + bz[n_i]) / 2)

bad_nodes = np.argsort(bres)[::-1][:4].tolist()

# lineage id per node, to pull each bad node's full track
bdaughters = set(btgt)
broots = [i for i in range(len(bn)) if i not in bdaughters]
blineage = np.full(len(bn), -1)
for lid, root in enumerate(broots):
    stack = [root]
    while stack:
        i = stack.pop()
        blineage[i] = lid
        stack.extend(bkids.get(i, []))

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for ax, i in zip(axes.flat, bad_nodes, strict=True):
    m = blineage == blineage[i]
    order = np.argsort(bt[m])
    ax.plot(bt[m][order], bz[m][order], ".-", color="C1", label="track")
    ax.plot(bt[i], bz[i], "o", color="red", markersize=10, label="mismatched node")
    ax.annotate(
        f"+{bres[i]:.0f} slices at t={bt[i]}",
        (bt[i], bz[i]),
        textcoords="offset points",
        xytext=(-70, -16),
        color="red",
    )
    ax.set_title(f"lineage {blineage[i]} — node {bn['node_id'][i]}", fontsize=10)
    ax.set_xlabel("t")
    ax.set_ylabel("z (slice)")

axes.flat[0].legend()
fig.suptitle("6bba_f20478e9: four one-frame identity swaps at t=65 (z spikes of +29..37 slices)")
plt.show()

In [ ]:
# the four mismatched tracks overlaid on the raw frames
# left = top view (Z-MIP), right = side view (X-MIP, z vertical)
# the side view exposes the t=65 z-spike
# red ring = the mismatched node, circle = current position
T_LO, T_HI = 45, 99
frames_t = np.arange(T_LO, T_HI + 1)
broot = zarr.open_group(DATA_DIR / "train" / "6bba_f20478e9.zarr", mode="r")
bvol = broot["0"]
assert isinstance(bvol, zarr.Array)
tops = np.stack([np.asarray(bvol[t]).max(axis=0) for t in range(T_LO, T_HI + 1)])  # (F, Y, X)
sides = np.stack([np.asarray(bvol[t]).max(axis=2) for t in range(T_LO, T_HI + 1)])  # (F, Z, Y)
bx = bn["x"].to_numpy()
by = bn["y"].to_numpy()

# per lineage trail segments (both views) revealed at the later endpoint's t, + node times
lin_data: list[dict[str, Any]] = []
for j, i in enumerate(bad_nodes):
    m = blineage == blineage[i]
    order = np.argsort(bt[m])
    nts, xs, ys, zs = bt[m][order], bx[m][order], by[m][order], bz[m][order]
    lin_data.append(
        {
            "lid": blineage[i],
            "top_segs": [np.array([[xs[k], ys[k]], [xs[k + 1], ys[k + 1]]]) for k in range(len(nts) - 1)],
            "side_segs": [np.array([[ys[k], zs[k]], [ys[k + 1], zs[k + 1]]]) for k in range(len(nts) - 1)],
            "seg_cuts": np.searchsorted(nts[1:], frames_t, side="right"),
            "node_cuts": np.searchsorted(nts, frames_t, side="right"),
            "nts": nts,
            "xs": xs,
            "ys": ys,
            "zs": zs,
            "bad_xy": (bx[i], by[i]),
            "bad_yz": (by[i], bz[i]),
            "color": f"C{j}",
        }
    )

fig, (ax_top, ax_side) = plt.subplots(1, 2, figsize=(13, 6))
im_top = ax_top.imshow(tops[0], cmap="gray")
im_side = ax_side.imshow(sides[0], cmap="gray")
ax_top.set_title("top view (Z-MIP)")
ax_side.set_title("side view (X-MIP) — z vertical")
for ld in lin_data:
    for ax, key in ((ax_top, "top"), (ax_side, "side")):
        coll = LineCollection([], colors=ld["color"], linewidths=1.2)
        ax.add_collection(coll)
        head = ax.scatter([], [], s=60, facecolors="none", edgecolors=ld["color"], linewidths=2)
        ring_xy = ld["bad_xy"] if key == "top" else ld["bad_yz"]
        ax.plot(*ring_xy, "o", markersize=14, markerfacecolor="none", markeredgecolor="red")
        ld[f"{key}_coll"] = coll
        ld[f"{key}_head"] = head
    ax_top.plot([], [], color=ld["color"], label=f"lineage {ld['lid']}")

ax_top.legend(loc="upper right", fontsize=8)


def update_overlay(f: int) -> None:
    im_top.set_data(tops[f])
    im_side.set_data(sides[f])
    for ld in lin_data:
        k = ld["seg_cuts"][f]
        ld["top_coll"].set_segments(ld["top_segs"][:k])
        ld["side_coll"].set_segments(ld["side_segs"][:k])
        n = ld["node_cuts"][f]
        if n > 0:
            ld["top_head"].set_offsets([[ld["xs"][n - 1], ld["ys"][n - 1]]])
            ld["side_head"].set_offsets([[ld["ys"][n - 1], ld["zs"][n - 1]]])
        else:
            ld["top_head"].set_offsets(np.empty((0, 2)))
            ld["side_head"].set_offsets(np.empty((0, 2)))

    fig.suptitle(f"6bba_f20478e9 — t={frames_t[f]} (mismatch at t=65)")


anim_overlay = animation.FuncAnimation(fig, update_overlay, frames=len(frames_t), interval=200, blit=False)
plt.close(fig)
HTML(anim_overlay.to_jshtml())

In [ ]:
# quantify irregular jumps per embryo at p95 and p99
# inspecting how many tracks contain >= 1 jump and what % of such a track is jumps
rows: list[dict[str, Any]] = []
for embryo in embryos:
    steps = _steps(seg_um[embryo])
    n_tracks = len(set(lids[embryo].tolist()))
    for p in (95, 99):
        thr = float(np.percentile(steps, p))
        per_track = (
            pl.DataFrame({"lid": lids[embryo], "jump": steps >= thr})
            .group_by("lid")
            .agg(pl.len().alias("steps"), pl.col("jump").sum().alias("jumps"))
        )
        flagged = per_track.filter(pl.col("jumps") > 0).with_columns(
            (100 * pl.col("jumps") / pl.col("steps")).alias("pct_irregular")
        )
        mean_pct = flagged["pct_irregular"].mean()
        max_pct = flagged["pct_irregular"].max()
        assert isinstance(mean_pct, float)
        assert isinstance(max_pct, float)
        rows.append(
            {
                "embryo": embryo,
                "threshold": f"p{p}",
                "thr_um": round(thr, 2),
                "tracks": n_tracks,
                "tracks_with_jumps": flagged.height,
                "pct_tracks_flagged": round(100 * flagged.height / n_tracks, 1),
                "mean_pct_of_track_irregular": round(mean_pct, 1),
                "max_pct_of_track_irregular": round(max_pct, 1),
            }
        )

jump_summary = pl.DataFrame(rows)
jump_summary

### Division Census, Annotation Sparsity & Node-Count Metadata